# MultiVar OpenFOAM POD + SINDy

Este notebook testa se usar **mais variáveis do OpenFOAM** ajuda o SINDy.

Campos usados no teste:

```text
Ux, Uy, alpha.water, p_rgh
```

Pipeline:

```text
campos OpenFOAM multivariáveis -> normalização por canal -> POD/SVD -> coeficientes reduzidos -> SINDy -> integração livre
```

Este caso complementa:

- `POD_SINDy.ipynb`: POD apenas do campo usado anteriormente.
- `SAE_POD_SINDy.ipynb`: SAE + POD no latente.
- **Este notebook**: POD direto em múltiplas variáveis OpenFOAM normalizadas.


## 1. Importações, caminhos e funções auxiliares


In [ ]:
import numpy as np, json, csv, warnings, time
from pathlib import Path
from scipy.integrate import solve_ivp
from scipy.signal import savgol_filter
from sklearn.utils.extmath import randomized_svd
from sklearn.linear_model import Lasso
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
CASE_DIR=Path('/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000')
CODE_DIR=Path('/home/mlep-pc-ubuntu/OpenFOAM/mlep-pc-ubuntu-13/run/damBreakLaminar 10 000/Codigos damBreakLaminar 10.000')
OUT=CODE_DIR/'SINDy/resultados/multivar_pod_sindy_quick'; OUT.mkdir(parents=True,exist_ok=True)

def rel(a,b): return float(np.linalg.norm(a-b)/(np.linalg.norm(a)+1e-12))
def load_X(stride=10, variables=('Ux','Uy','alpha','p')):
    U=np.load(CASE_DIR/'rom_melhorias/cache/snapshots_U.npy', mmap_mode='r')[::stride]
    alpha=np.load(CASE_DIR/'rom_melhorias/cache/snapshots_alpha.water.npy', mmap_mode='r')[::stride]
    p=np.load(CASE_DIR/'rom_melhorias/cache/snapshots_p_rgh.npy', mmap_mode='r')[::stride]
    blocks=[]; channels=[]
    if 'Ux' in variables: blocks.append(np.asarray(U[:,:,0],np.float64)); channels.append('Ux')
    if 'Uy' in variables: blocks.append(np.asarray(U[:,:,1],np.float64)); channels.append('Uy')
    if 'Uz' in variables: blocks.append(np.asarray(U[:,:,2],np.float64)); channels.append('Uz')
    if 'alpha' in variables: blocks.append(np.asarray(alpha,np.float64)); channels.append('alpha.water')
    if 'p' in variables: blocks.append(np.asarray(p,np.float64)); channels.append('p_rgh')
    X=np.concatenate(blocks, axis=1)
    # normalize per block/channel
    start=0; stats=[]
    Xn=np.empty_like(X)
    ncell=alpha.shape[1]
    for b,ch in zip(blocks,channels):
        end=start+b.shape[1]
        mu=float(np.mean(X[:,start:end])); sig=float(np.std(X[:,start:end])); sig=max(sig,1e-12)
        Xn[:,start:end]=(X[:,start:end]-mu)/sig
        stats.append({'channel':ch,'mean':mu,'std':sig,'start':start,'end':end})
        start=end
    times=np.load(BASE/'POD/resultados/resultados_pod_completos.npz', allow_pickle=True)['times'][::stride].astype(float)
    return Xn,times,stats

def powers(n,order):
    if n==1: return [(i,) for i in range(order+1)]
    out=[]
    for p in range(order+1):
        for suf in powers(n-1,order-p): out.append((p,)+suf)
    return sorted(out)
def theta(X,order=1):
    P=powers(X.shape[1],order); T=np.empty((X.shape[0],len(P)))
    for j,p in enumerate(P): T[:,j]=np.prod(np.power(X,p),axis=1)
    return T,P
def fit_stlsq(T,dX,lam,max_iter=10):
    Xi=np.linalg.lstsq(T,dX,rcond=None)[0]
    for _ in range(max_iter):
        small=np.abs(Xi)<lam; Xi[small]=0
        for k in range(dX.shape[1]):
            big=~small[:,k]
            if big.any(): Xi[big,k]=np.linalg.lstsq(T[:,big],dX[:,k],rcond=None)[0]
            else: Xi[:,k]=0
    return Xi
def sim(Tfun,Xi,t,x0):
    def rhs(_,x): return Tfun(x)@Xi
    sol=solve_ivp(rhs,(float(t[0]),float(t[-1])),x0,t_eval=t,rtol=1e-6,atol=1e-8,method='RK45')
    if not sol.success or sol.y.shape[1]!=len(t): return None, sol.message
    return sol.y.T,None



## 2. Executar busca multivariável

Por padrão usa `stride=10` para reduzir custo computacional. Para usar todos os snapshots, altere para `stride=1`, mas o custo aumenta bastante.


In [ ]:
t0=time.time(); X,t,stats=load_X(stride=10, variables=('Ux','Uy','alpha','p')); print('X',X.shape,'load',time.time()-t0)
Xmean=X.mean(axis=0); Xc=X-Xmean
U,s,Vt=randomized_svd(Xc, n_components=16, n_iter=5, random_state=1)
energy=np.cumsum(s*s)/np.sum(s*s) # approximate captured among 16 only not total; compute norm total
energy_total=np.sum(s*s)/np.linalg.norm(Xc,'fro')**2
print('svd done',time.time()-t0,'energy16',energy_total)
rows=[]
for modes in [4,8,12,16]:
    A=U[:,:modes]*s[:modes]
    # scale coeffs
    am=A.mean(0); ast=A.std(0); ast[ast<1e-12]=1
    Z=(A-am)/ast
    for smooth in [0,21,51]:
      Zfit=Z if smooth==0 else savgol_filter(Z, smooth if smooth%2 else smooth+1, 3, axis=0)
      dZ=np.gradient(Zfit,t,axis=0,edge_order=2)
      for order in [1,2]:
        T,P=theta(Zfit,order)
        for lam in [1e-3,1e-2,1e-1,3e-1,1.0]:
          Xi=fit_stlsq(T,dZ,lam)
          dPred=T@Xi
          # one step
          pred=np.zeros_like(Z); pred[0]=Z[0]
          def tfun(x): return theta(np.atleast_2d(x),order)[0]
          for i in range(1,len(t)):
            pred[i]=Z[i-1]+(t[i]-t[i-1])*(tfun(Z[i-1:i])@Xi)
          free,msg=sim(tfun,Xi,t,Z[0])
          free_err=None if free is None else rel(Z,free)
          row=dict(modes=modes,smooth=smooth,order=order,alpha=lam,features=T.shape[1],nnz=int(np.count_nonzero(np.abs(Xi)>1e-10)),deriv=rel(dZ,dPred),one=rel(Z,pred),free_success=free is not None,free=free_err,msg=msg)
          rows.append(row); print(row)
with (OUT/'search.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
best=min(rows,key=lambda r: (0 if r['free_success'] else 1000)+(r['free'] if r['free'] is not None else 999)+0.1*r['deriv'])
(OUT/'summary.json').write_text(json.dumps({'best':best,'n':len(rows),'stats':stats,'stride':10,'variables':['Ux','Uy','alpha.water','p_rgh'],'svd_energy16_est':float(energy_total)},indent=2))
print('BEST',best)


## 3. Ler resumo salvo


In [ ]:
summary = json.loads((OUT / 'summary.json').read_text())
summary
